# eBay Laptops & Netbooks - Modeling

In this project, we aim to build a predictive model for **estimating laptop prices** using a dataset which contains cleaned information from eBay's Laptops & Netbooks category, originally obtained via web scraping, which includes product attributes such as brand, specifications, and other listing details.

The model utilizes **CatBoost** (*Categorical Boosting*), a state-of-the-art gradient boosting library [developed by Yandex][Yandex CatBoost], renowned for its efficiency in handling categorical features and its strong performance in regression, classification & ranking.

<div align="center">
<img src="../assets/logos/catboost_logo.png" height="225" width="225"/>
</div>

[Yandex Catboost]: https://yandex.com/dev/catboost/

## Links & Information

**Project Repository** - GitHub: [Laptop Price Prediction with CatBoost][Project Code]

[Project Code]: https://github.com/jxareas/laptop-price-catboost


### Importing Libraries

In this project, we will leverage several powerful libraries for efficient data manipulation, feature engineering, modeling, and hyperparameter tuning.

- **Scikit-learn**: Used for **training the model**, performing model evaluation, and splitting the dataset in train, validation & test sets.
- **CatBoost**: The **core library** for modeling, used to design a gradient boosting model to handle categorical features effectively and provide good performance in regression tasks.
- **Optuna**: Utilized for **hyperparameter tuning**, automating the process of finding the best model parameters to improve performance.
- **Feature Engine**: A library for **feature engineering** that provides various techniques for transforming and selecting features, ensuring that our model has the most relevant data.
- **Seaborn**: Used for **exploratory data analysis (EDA)** and plotting, providing intuitive and high-level visualizations to understand the dataset and its relationships.
- **SHAP**: Used for **model interpretability**, providing insights into how each feature influences the model's predictions, helping to explain the decisions made by the model & ensuring transparency / Explainable AI.

These libraries will work together to ensure a streamlined and efficient manner for building and optimizing the predictive model.


In [ ]:
# Importing libraries and setting constants

# Libraries
import polars as pl
from feature_engine.encoding import RareLabelEncoder

# Constants
RANDOM_SEED = 287
DATA_SOURCE_PATH = '../data/ebay_laptops_and_netbooks_cleansed.csv'


## Data Preparation

### Loading the dataset

In [ ]:
df = pl.read_csv(DATA_SOURCE_PATH)
df.head(n=10)

## Exploratory Data Analysis

In [ ]:
# TODO : Exploratory Data Analysis

## Feature Engineering


In [ ]:
# The target variable : the minimum pricer required to purchase the item (laptop/netbook)
target_var = 'min_price'
# A polars expression for feature selection -> selecting every column except for the target variable & currency (currency cardinality is one)
feature_selection_expr = pl.all().exclude(target_var, 'currency', 'seller_note')

# Dataframe holding all feature variables
df_features = df.select(
    feature_selection_expr
)
# Series holding the target vector
series_target = df[target_var]

### Defining the feature names

In [ ]:
feature_names = df_features.columns
cat_feature_names = df_features.select(
    pl.col(pl.String),
).columns

### Encoding rare labels

In this project, we leverage the [**Feature-engine**][Feature-Engine] package to handle categorical variables efficiently, specifically using `RareLabelEncoder` to group infrequent categories under a common label.

Rare categories in a dataset can introduce **high cardinality**, making statistical analysis and model generalization more challenging. By setting a **minimum frequency threshold**, we ensure that only sufficiently common categories remain distinct, while rare ones are grouped into an "other" category.

This reduces noise, prevents overfitting, and enhances model interpretability without significantly losing information.

<div align="center">
<img src="../assets/logos/feature_engine.png" height="125" width="125"/>
</div>

[Feature-Engine]: https://github.com/feature-engine/feature_engine

In [ ]:
df_features.head(n=10).to_pandas()

In [ ]:
# Sets the minimum count for a category to be kept separately, categories with fewer than 20 occurrences will be grouped.
MIN_COUNT_FOR_LABEL = 20
# Sets the tolerance for rare categories based on the minimum count and total number of rows in the dataset.
TOLERANCE_FOR_LABEL = MIN_COUNT_FOR_LABEL / df_features.height

encoder = RareLabelEncoder(n_categories=1, replace_with='other', tol=TOLERANCE_FOR_LABEL)
for col in cat_feature_names:
    encoder_transform = encoder.fit_transform(df_features[[col]].fill_null('unknown').to_pandas())
    df_features = df_features.with_columns(
        pl.Series(encoder_transform[col])
    )
    display(df_features[col].value_counts(sort=True, parallel=True))

## Machine Learning

In [ ]:
# TODO : Machine Learning

### Splitting the data - Train-Validation-Test Split

<div align="center">
<img src="../assets/images/train_test_validation_split.png" height="375" width="375"/>
</div>

### Categorical Boosting

**CatBoost** is a gradient boosting algorithm that builds decision trees sequentially, improving predictions at each step (*Ensemble learning*).

A key feature is how CatBoost handles **categorical** data (hence its name). Instead of one-hot encoding or label encoding, it uses **target encoding**, where categories are replaced by smoothed target statistics, essentially *converting categorical values into numerical values*.

Additionally, trees CatBoost builds are **oblivious trees**, meaning that every split at a given depth applies the same condition across all nodes. This structure helps prevent overfitting, improves generalization, and makes computation highly efficient, especially on GPUs.

Overall, CatBoost refines the traditional gradient boosting approach by handling categorical data naturally, and optimizing computation, making it one of the best choices for tabular data problems with a considerable amount of categorical variables.

<div align="center">
<img src="../assets/images/categorical_boosting.png" height="375" width="375"/>
<br>
<a href="https://www.researchgate.net/figure/The-flow-diagram-of-the-CatBoost-model_fig3_370695897"><i>The flow diagram of the CatBoost model</i></a>
</div>

## Hyperparameter Tuning

In [ ]:
# TODO : Hyperparameter Tuning

## Explainable AI - SHAP

In [ ]:
# TODO: Explainable AI - SHAP